This implementation performs a fully scalable, generic systolic array architecture that can be compiled to any size.
The key to this flexibility is the use of genvar syntax combined with nested generate loops in the structural RTL (sysarray.v). Instead of manually copying, pasting, and hard-coding processing elements, the compiler calculates the grid layout dynamically during elaboration.

How to Change the Array Size:

1. Modify the RTL Parameters: By simply changing the values of the ROW and COL parameters inside your top-level module instantiation, the hardware description compiler will instantly generate the appropriate array grid size (e.g., resizing from 4 × 4 to 8 × 8 or 16 × 16).
2. Update the Testbench: When changing the structural size of the array, the testbench must be updated to match the chosen ROW and COL bounds. This ensures that:
	• The matrix ingestion timelines are properly skewed (Total Steps = 2 × (ROW - 1) + Length).
	• The physical data injection wires map correctly to the updated external boundaries of the matrix grid.


In [17]:
!apt-get install iverilog


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (12.0-2build2).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [18]:
%%writefile PE.v
module PE #(
    parameter DATA_WIDTH = 16,
    parameter ACC_WIDTH  = 32
)(
    input wire clk, rst,
    input wire signed [DATA_WIDTH-1:0] a_in, b_in,
    output reg signed [DATA_WIDTH-1:0] a_out, b_out,
    output reg signed [ACC_WIDTH-1:0]  c_out
);

always @(posedge clk or posedge rst)
begin
    if (rst == 1'b1)
    begin
        a_out <= 0;
        b_out <= 0;
        c_out <= 0;
    end
    else
    begin
        a_out <= a_in;
        b_out <= b_in;
        c_out <= c_out + (a_in * b_in);
    end
end

endmodule


Overwriting PE.v


In [19]:
%%writefile sysarray.v
module SystolicArrayMod #(
    parameter DATA_WIDTH = 16,
    parameter ACC_WIDTH  = 32,
    parameter ROW = 4,
    parameter COL = 4
)(
    input wire Clk, Rst,
    input wire signed [DATA_WIDTH-1:0] A_row0, A_row1, A_row2, A_row3,
    input wire signed [DATA_WIDTH-1:0] B_col0, B_col1, B_col2, B_col3,
    output wire signed [ACC_WIDTH-1:0] C_00, C_01, C_02, C_03, C_10, C_11, C_12, C_13, C_20, C_21, C_22, C_23, C_30, C_31, C_32, C_33
);

    // Internal wire arrays for interconnects
    wire signed [DATA_WIDTH-1:0] A [0:ROW-1][0:COL];
    wire signed [DATA_WIDTH-1:0] B [0:ROW][0:COL-1];
    wire signed [ACC_WIDTH-1:0]  C [0:ROW-1][0:COL-1];

    // Explicitly mapping Row Inputs to Row Index 0 of the internal A array
    assign A[0][0] = A_row0;
    assign A[1][0] = A_row1;
    assign A[2][0] = A_row2;
    assign A[3][0] = A_row3;

    // Explicitly mapping Column Inputs to Column Index 0 of the internal B array
    assign B[0][0] = B_col0;
    assign B[0][1] = B_col1;
    assign B[0][2] = B_col2;
    assign B[0][3] = B_col3;

    // Mapping internal Accumulator results out to top-level individual port wires
    assign C_00 = C[0][0]; assign C_01 = C[0][1]; assign C_02 = C[0][2]; assign C_03 = C[0][3];
    assign C_10 = C[1][0]; assign C_11 = C[1][1]; assign C_12 = C[1][2]; assign C_13 = C[1][3];
    assign C_20 = C[2][0]; assign C_21 = C[2][1]; assign C_22 = C[2][2]; assign C_23 = C[2][3];
    assign C_30 = C[3][0]; assign C_31 = C[3][1]; assign C_32 = C[3][2]; assign C_33 = C[3][3];

    // Generate loop for the 4x4 matrix processing elements
    genvar r, c;
    generate
        for (r=0; r<ROW; r=r+1) begin: genrow
            for (c=0; c<COL; c=c+1) begin: gencol
                PE PE_inst (
                    .clk   (Clk),
                    .rst   (Rst),
                    .a_in  (A[r][c]),
                    .b_in  (B[r][c]),
                    .a_out (A[r][c+1]),
                    .b_out (B[r+1][c]),
                    .c_out (C[r][c])
                );
            end
        end
    endgenerate

endmodule


Overwriting sysarray.v


In [20]:
%%writefile tb.v
module testbench;

    parameter DATA_WIDTH = 16;
    parameter ACC_WIDTH  = 32;
    parameter ROW        = 4;
    parameter COL        = 4;
    parameter Length     = 4;

    reg clk_test;
    reg rst_test;

    reg signed [DATA_WIDTH-1:0] A_row_test [0:ROW-1];
    reg signed [DATA_WIDTH-1:0] B_col_test [0:COL-1];

    wire signed [ACC_WIDTH-1:0] C_out_00, C_out_01, C_out_02, C_out_03;
    wire signed [ACC_WIDTH-1:0] C_out_10, C_out_11, C_out_12, C_out_13;
    wire signed [ACC_WIDTH-1:0] C_out_20, C_out_21, C_out_22, C_out_23;
    wire signed [ACC_WIDTH-1:0] C_out_30, C_out_31, C_out_32, C_out_33;

    reg signed [DATA_WIDTH-1:0] A [0:ROW-1][0:Length-1];
    reg signed [DATA_WIDTH-1:0] B [0:Length-1][0:COL-1];

    integer t, r, c;
    integer col_idx, row_idx;

    SystolicArrayMod #(
        .DATA_WIDTH(DATA_WIDTH),
        .ACC_WIDTH(ACC_WIDTH),
        .ROW(ROW),
        .COL(COL)
    ) dut (
        .Clk(clk_test),
        .Rst(rst_test),
        .A_row0(A_row_test[0]), .A_row1(A_row_test[1]), .A_row2(A_row_test[2]), .A_row3(A_row_test[3]),
        .B_col0(B_col_test[0]), .B_col1(B_col_test[1]), .B_col2(B_col_test[2]), .B_col3(B_col_test[3]),
        .C_00(C_out_00), .C_01(C_out_01), .C_02(C_out_02), .C_03(C_out_03),
        .C_10(C_out_10), .C_11(C_out_11), .C_12(C_out_12), .C_13(C_out_13),
        .C_20(C_out_20), .C_21(C_out_21), .C_22(C_out_22), .C_23(C_out_23),
        .C_30(C_out_30), .C_31(C_out_31), .C_32(C_out_32), .C_33(C_out_33)
    );

    always #10 clk_test = ~clk_test;

    initial begin
        clk_test = 0;
        rst_test = 1;

        for (r = 0; r < ROW; r = r + 1) begin
            A_row_test[r] = {DATA_WIDTH{1'b0}};
        end
        for (c = 0; c < COL; c = c + 1) begin
            B_col_test[c] = {DATA_WIDTH{1'b0}};
        end

        // Generate known input matrices
        for (r = 0; r < ROW; r = r + 1) begin
            for (c = 0; c < Length; c = c + 1) begin
                A[r][c] = r + c + 1;
                B[c][r] = c + r + 2;
            end
        end

        // Print input matrices first
        $display("\n=== INPUT MATRIX A ===");
        for (r = 0; r < ROW; r = r + 1) begin
            $display("[%3d, %3d, %3d, %3d]", A[r][0], A[r][1], A[r][2], A[r][3]);
        end

        $display("\n=== INPUT MATRIX B ===");
        for (r = 0; r < Length; r = r + 1) begin
            $display("[%3d, %3d, %3d, %3d]", B[r][0], B[r][1], B[r][2], B[r][3]);
        end

        #40;
        rst_test = 0; // Release Reset

        // Feeding data in step-by-step with a skew factor
        for (t = 0; t < ((2 * (ROW - 1)) + Length); t = t + 1) begin
            @ (negedge clk_test);

            for (r = 0; r < ROW; r = r + 1) begin
                col_idx = t - r;
                if (col_idx >= 0 && col_idx < Length) begin
                    A_row_test[r] = A[r][col_idx];
                end else begin
                    A_row_test[r] = {DATA_WIDTH{1'b0}};
                end
            end

            for (c = 0; c < COL; c = c + 1) begin
                row_idx = t - c;
                if (row_idx >= 0 && row_idx < Length) begin
                    B_col_test[c] = B[row_idx][c];
                end else begin
                    B_col_test[c] = {DATA_WIDTH{1'b0}};
                end
            end
        end

        // Wait for last element computation to settle inside PE arrays
        @(negedge clk_test);
        #20;

        $display("\n=== SIMULATION COMPLETE! OUTPUT MATRIX C ===");
        $display("[%5d, %5d, %5d, %5d]", C_out_00, C_out_01, C_out_02, C_out_03);
        $display("[%5d, %5d, %5d, %5d]", C_out_10, C_out_11, C_out_12, C_out_13);
        $display("[%5d, %5d, %5d, %5d]", C_out_20, C_out_21, C_out_22, C_out_23);
        $display("[%5d, %5d, %5d, %5d]", C_out_30, C_out_31, C_out_32, C_out_33);
        $display("===========================================\n");
        $finish;
    end

endmodule


Overwriting tb.v


In [21]:
!apt-get install -y iverilog



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (12.0-2build2).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [22]:
!iverilog -o systolic_sim PE.v sysarray.v tb.v


In [23]:
!vvp systolic_sim



=== INPUT MATRIX A ===
[  1,   2,   3,   4]
[  2,   3,   4,   5]
[  3,   4,   5,   6]
[  4,   5,   6,   7]

=== INPUT MATRIX B ===
[  2,   3,   4,   5]
[  3,   4,   5,   6]
[  4,   5,   6,   7]
[  5,   6,   7,   8]

=== SIMULATION COMPLETE! OUTPUT MATRIX C ===
[   40,    50,    60,    70]
[   54,    68,    82,    96]
[   68,    86,   104,   122]
[   82,   104,   126,   148]

tb.v:110: $finish called at 260 (1s)
